# Phishing Email Detection Pipeline (Enron-Spam)

This notebook covers the complete NLP classification pipeline for identifying phishing/spam emails, including:
1. **Dataset Download & Setup**: Fetching and extracting the Enron-Spam dataset.
2. **Data Cleansing**: Merging Subject & Body text, resolving null values, and labeling classifications.
3. **TF-IDF Vectorization**: Extracting features using TF-IDF representation (unigrams & bigrams).
4. **Logistic Regression Classifier**: Training, evaluating, and extracting feature coefficients for explainability (identifying key spam words).

## 1. Environment Setup & Data Fetching

In [ ]:
import os
import time
import zipfile
import urllib.request
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Set up directories relative to the notebook path
NOTEBOOK_DIR = Path(os.getcwd())
ROOT_DIR = NOTEBOOK_DIR.parent
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
MODELS_DIR = ROOT_DIR / "data" / "models"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Download and unzip Enron Spam dataset if not present
zip_path = RAW_DIR / "enron_spam.zip"
csv_path = RAW_DIR / "enron_spam_data.csv"

if not csv_path.exists():
    print("Downloading enron_spam_data.zip...")
    urllib.request.urlretrieve(
        "https://github.com/MWiechmann/enron_spam_data/raw/master/enron_spam_data.zip", 
        zip_path
    )
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(RAW_DIR)
    print("Enron dataset ready!")
    if zip_path.exists():
        os.remove(zip_path)
else:
    print("enron_spam_data.csv already exists locally.")

## 2. Preprocessing & Splitting

In [ ]:
print("Loading Enron spam dataset...")
df = pd.read_csv(csv_path)
print(f"Raw dataset shape: {df.shape}")

df["Subject"] = df["Subject"].fillna("")
df["Message"] = df["Message"].fillna("")
df["text"] = (df["Subject"] + " " + df["Message"]).str.strip()

# Drop empty records
df = df[df["text"].str.len() > 0].reset_index(drop=True)
print(f"Shape after removing empty messages: {df.shape}")

# Label Ham/Spam (Spam = 1, Ham = 0)
df["label"] = (df["Spam/Ham"].str.lower() == "spam").astype(int)
print("\nClass balance:")
print(df["label"].value_counts(normalize=True))

In [ ]:
# Plot the class balance
plt.figure(figsize=(6, 3.5))
sns.countplot(x="label", data=df, palette="Accent")
plt.title("Class Distribution (0: Ham, 1: Spam/Phishing)")
plt.xticks([0, 1], ["Ham (Clean)", "Spam (Phishing)"])
plt.show()

In [ ]:
# Split into Train and Test
train_df, test_df = train_test_split(
    df[["text", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

train_df.to_csv(PROCESSED_DIR / "email_train.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "email_test.csv", index=False)

print(f"Train set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")

## 3. TF-IDF Vectorization & Model Training

In [ ]:
print("Fitting TF-IDF Vectorizer (unigrams + bigrams)...")
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words="english",
    sublinear_tf=True,
)

t0 = time.time()
X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])
print(f"Vectorized in {time.time() - t0:.2f}s, Vocabulary size: {len(vectorizer.vocabulary_)}")

y_train = train_df["label"].values
y_test = test_df["label"].values

In [ ]:
print("Training Logistic Regression model...")
clf = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", random_state=42)

t0 = time.time()
clf.fit(X_train, y_train)
print(f"Model trained in {time.time() - t0:.2f}s")

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

## 4. Evaluation & Interpretability

In [ ]:
print("=== Phishing Classifier Evaluation on Test Set ===")
print(classification_report(y_test, y_pred, target_names=["ham", "spam/phishing"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

In [ ]:
# Explainability: Extract top tokens predicting Spam/Phishing vs Clean Ham
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = clf.coef_[0]

top_spam_idx = np.argsort(coefs)[::-1][:15]
top_ham_idx = np.argsort(coefs)[:15]

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.barplot(x=coefs[top_spam_idx], y=feature_names[top_spam_idx], palette="Reds_r")
plt.title("Top Words Indicating Spam/Phishing")
plt.xlabel("Coefficient (Importance Weight)")

plt.subplot(1, 2, 2)
sns.barplot(x=np.abs(coefs[top_ham_idx]), y=feature_names[top_ham_idx], palette="Greens_r")
plt.title("Top Words Indicating Ham (Legitimate)")
plt.xlabel("Absolute Coefficient (Importance Weight)")

plt.tight_layout()
plt.show()

In [ ]:
# Dump model objects to joblib
joblib.dump(vectorizer, MODELS_DIR / "email_tfidf_vectorizer.joblib")
joblib.dump(clf, MODELS_DIR / "email_classifier.joblib")
print(f"Saved Vectorizer and Classifier models in: {MODELS_DIR}")